In [ ]:
"""
===============================================================================
VISIBILITY INTO MARKET EVENTS
===============================================================================

Question

    Can we observe the events that change the market?

Answer

    Yes, but the level of visibility depends on
    the data feed provided by the exchange.

===============================================================================
1. NEW ORDER ARRIVES
===============================================================================

Visibility

    ✓ Yes

How?

    • Order Book Snapshot

    • Order Book WebSocket

Observation

    A new price level appears

or

    Quantity increases at an existing level.

Example

Before

    $1855.40

    Quantity = 5 ETH

After

    $1855.40

    Quantity = 20 ETH

Inference

    Someone added liquidity.

===============================================================================
2. ORDER CANCELLED
===============================================================================

Visibility

    ✓ Yes

How?

    Compare consecutive order book updates.

Observation

Before

    $1855.40

    Quantity = 20 ETH

After

    $1855.40

    Quantity = 3 ETH

Inference

    Liquidity disappeared.

Question

    Was it cancelled?

or

    Was it executed?

The order book alone cannot always tell.

===============================================================================
3. ORDER MODIFIED
===============================================================================

Visibility

    ✓ Indirectly

Observation

Quantity changes

or

Price changes

Inference

    An existing order was likely modified.

===============================================================================
4. TRADE EXECUTED
===============================================================================

Visibility

    ✓ Yes

Requires

    Trade Stream

or

    Recent Trades API

Observation

    • Execution Price

    • Quantity

    • Timestamp

This represents completed transactions.

===============================================================================
"""

In [ ]:
"""
================================================================================
ORDER BOOK RESEARCH DATASET
================================================================================

For every observation, record

    • Local Timestamp

    • Exchange Timestamp

    • Best Bid

    • Best Ask

    • Full Order Book

    • Executed Trades

Later compute

    • Bid Volume

    • Ask Volume

    • Liquidity Imbalance

    • Spread

    • Mid Price

    • Price Evolution

    • Liquidity Evolution

Objective

    Reconstruct the sequence of market events
    and study how participant decisions
    evolve into price movement.

================================================================================
"""

In [ ]:
"""
===============================================================================
MEXC Order Book Snapshot with Timestamps
===============================================================================

Objective

    Retrieve the current order book together with

        • Local Timestamp (UTC)
        • Exchange Timestamp (UTC)

This is the foundation for building our historical
order book dataset.

Endpoints

    GET /api/v3/time
    GET /api/v3/depth

===============================================================================
"""

from datetime import datetime, timezone
import requests

BASE_URL = "https://api.mexc.com"

# ==============================================================================
# Retrieve Exchange Timestamp
# ==============================================================================

time_response = requests.get(f"{BASE_URL}/api/v3/time")
time_response.raise_for_status()

exchange_time_ms = time_response.json()["serverTime"]

# Convert milliseconds -> UTC datetime
exchange_time = datetime.fromtimestamp(
    exchange_time_ms / 1000,
    tz=timezone.utc
)

# ==============================================================================
# Record Local Timestamp
# ==============================================================================

# Store local machine time as UTC
local_time = datetime.now(timezone.utc)

# ==============================================================================
# Retrieve Current Order Book
# ==============================================================================

params = {
    "symbol": "ETHUSDT",
    "limit": 10
}

response = requests.get(
    f"{BASE_URL}/api/v3/depth",
    params=params
)

response.raise_for_status()

orderbook = response.json()

# ==============================================================================
# Store Snapshot
# ==============================================================================

snapshot = {
    "local_time_utc": local_time,
    "exchange_time_utc": exchange_time,
    "orderbook": orderbook
}



# ==============================================================================
# Display Buy Orders (Bids)
# ==============================================================================

print("=" * 70)
print("TOP BUY ORDERS (BIDS)")
print("=" * 70)

print(f"{'Price':>12} {'Quantity':>15} {'Timestamp (UTC)':>30}")

for price, quantity in snapshot["orderbook"]["bids"]:
    print(
        f"{price:>12} "
        f"{quantity:>15} "
        f"{snapshot['exchange_time_utc']}"
    )

print()

# ==============================================================================
# Display Sell Orders (Asks)
# ==============================================================================

print("=" * 70)
print("TOP SELL ORDERS (ASKS)")
print("=" * 70)

print(f"{'Price':>12} {'Quantity':>15} {'Timestamp (UTC)':>30}")

for price, quantity in snapshot["orderbook"]["asks"]:
    print(
        f"{price:>12} "
        f"{quantity:>15} "
        f"{snapshot['exchange_time_utc']}"
    )

TOP BUY ORDERS (BIDS)
       Price        Quantity                Timestamp (UTC)
     1854.67        30.02922 2026-07-25 09:24:31.864000+00:00
     1854.66         0.05082 2026-07-25 09:24:31.864000+00:00
     1854.65         0.05072 2026-07-25 09:24:31.864000+00:00
     1854.64         0.05214 2026-07-25 09:24:31.864000+00:00
     1854.63         0.05379 2026-07-25 09:24:31.864000+00:00
     1854.54        32.05028 2026-07-25 09:24:31.864000+00:00
     1854.47         0.00300 2026-07-25 09:24:31.864000+00:00
     1854.41        12.93971 2026-07-25 09:24:31.864000+00:00
     1854.38        34.03722 2026-07-25 09:24:31.864000+00:00
     1854.37        12.93804 2026-07-25 09:24:31.864000+00:00

TOP SELL ORDERS (ASKS)
       Price        Quantity                Timestamp (UTC)
     1854.68         3.15407 2026-07-25 09:24:31.864000+00:00
     1854.76         3.15407 2026-07-25 09:24:31.864000+00:00
     1854.80        12.93971 2026-07-25 09:24:31.864000+00:00
     1854.81         3.15407